# MicroCoder — Kaggle training
Tiny byte-level Python specialist with continual-feedback support.


In [ ]:
!pip -q install "datasets>=3.0"
import os, json, sys, shutil
from pathlib import Path
ROOT=Path('/kaggle/working/microcoder')
ROOT.mkdir(parents=True, exist_ok=True)
print(ROOT)


## 1. Download the Python dataset from Hugging Face


In [ ]:
from datasets import load_dataset
repo='youmyron/bits-py-dataset'
ds=load_dataset(repo, revision='main')
for split in ds:
    print(split, len(ds[split]))


In [ ]:
DATA=ROOT/'data'; DATA.mkdir(exist_ok=True)
for split in ds:
    target=DATA/f'{split}.jsonl'
    ds[split].to_json(target, orient='records', lines=True, force_ascii=False)
    print(target, target.stat().st_size)


## 2. Pull canonical code from GitHub


In [ ]:
!git clone -q https://github.com/Musabgpt/gpt.git /kaggle/working/gpt || true
%cd /kaggle/working/gpt
!git rev-parse HEAD


## 3. Train MicroCoder-284K


In [ ]:
import subprocess, sys
cmd=[sys.executable,'src/microcoder/train.py','--data',str(DATA/'train.jsonl'),'--config','configs/microcoder_284k.json','--out','/kaggle/working/outputs','--steps','20000','--batch-size','32','--grad-accum','2']
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 4. Optional feedback cycle with replay


In [ ]:
feedback=Path('/kaggle/working/feedback.jsonl')
if feedback.exists():
    cmd=[sys.executable,'src/microcoder/feedback.py','--checkpoint','/kaggle/working/outputs/best.pt','--feedback',str(feedback),'--mode','dpo','--out','/kaggle/working/outputs/feedback.pt']
    subprocess.run(cmd, check=True)
else:
    print('No feedback.jsonl yet; base training only.')


## 5. Persist
Download validated outputs or copy them into `/gpt/TinyPyGPT/export` in Google Drive.
